# LAB 10 — Tokenizer và answer span alignment

Mục tiêu: xem question/context được ghép thành input, kiểm tra `input_ids`, `attention_mask`, `token_type_ids` nếu tokenizer có, `sequence_ids`, `offset_mapping` và các cửa sổ tràn. Chạy từ thư mục `lab10/` để import `qa_model.py`. Tự làm bài tính trong `calculations.md`; notebook này chỉ kiểm tra bằng code.

In [ ]:
%pip -q install 'transformers>=4.45,<5' 'datasets>=3,<5'

In [ ]:
from transformers import AutoTokenizer
from qa_model import prepare_train_features

MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 128
STRIDE = 32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "Offset mapping cần fast tokenizer"
print("Tokenizer:", tokenizer.name_or_path)
print("Model inputs:", tokenizer.model_input_names)

## 1. Quan sát một sample

Đầu vào là question, context và answer text. Cell tự tìm vị trí character của answer, in token tương ứng và offsets để đối chiếu. `token_type_ids` chỉ xuất hiện nếu tokenizer hỗ trợ.

In [ ]:
context = "The annual summit was held in New York City before moving to Boston."
question = "Where was the annual summit held?"
answer = "New York City"
answer_start = context.index(answer)
encoded = tokenizer(question, context, return_offsets_mapping=True)
print("answer_start (character):", answer_start)
print("input_ids:", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
print("token_type_ids:", encoded.get("token_type_ids", "Tokenizer không sử dụng"))
for i, (token, part, offset) in enumerate(zip(tokenizer.convert_ids_to_tokens(encoded["input_ids"]), encoded.sequence_ids(), encoded["offset_mapping"])):
    print(f"{i:>3} {str(part):>4} {token:<15} {offset}")

## 2. Ánh xạ character span sang token span

Hàm từ `qa_model.py` xử lý span nằm giữa token, window không chứa answer và các offset ngoài context. Kiểm tra dữ liệu gốc trước khi diễn giải kết quả.

In [ ]:
examples = {"id": ["demo"], "question": [question], "context": [context],
            "answers": [{"text": [answer], "answer_start": [answer_start]}]}
features = prepare_train_features(examples, tokenizer, MAX_LENGTH, STRIDE)
for i, (start, end) in enumerate(zip(features["start_positions"], features["end_positions"])):
    tokens = tokenizer.convert_ids_to_tokens(features["input_ids"][i])
    print("window", i, "start token", start, "end token", end,
          "token span", tokens[start:end + 1])

## 3. Truncation và sliding window

Context lặp dài tạo nhiều window. `overflow_to_sample_mapping` cho biết mỗi feature đến từ sample nào; các window không chứa toàn bộ answer được gán CLS khi huấn luyện.

In [ ]:
long_context = "Background information. " * 80 + "The answer is silver oak. " + "More background information. " * 30
long_answer = "silver oak"
long_examples = {"id": ["long"], "question": ["What is the answer?"],
                 "context": [long_context],
                 "answers": [{"text": [long_answer], "answer_start": [long_context.index(long_answer)]}]}
windows = tokenizer(long_examples["question"], long_examples["context"],
                    truncation="only_second", max_length=MAX_LENGTH, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True)
labels = prepare_train_features(long_examples, tokenizer, MAX_LENGTH, STRIDE)
print("Số window:", len(windows["input_ids"]))
print("overflow_to_sample_mapping:", windows["overflow_to_sample_mapping"])
for i in range(len(windows["input_ids"])):
    cls = labels["input_ids"][i].index(tokenizer.cls_token_id)
    print(i, "start/end:", labels["start_positions"][i], labels["end_positions"][i],
          "answer trong window:", labels["start_positions"][i] != cls)

## 4. Trường hợp cần tự kiểm tra

Hãy thử thay answer bằng một chuỗi bắt đầu/kết thúc giữa token; quan sát token span được chọn. Thử đặt answer ở ngoài window đầu. Giải thích bằng lời sự khác nhau giữa character index, word index và token index trong `calculations.md` của bạn.